# 04b — Social Media Charts (Altair + vl-convert)

Publication-ready PNG charts using the @unwelcomedata brand palette.
All charts export to twitter_landscape (1600×900px) with watermark.

**4 Production Charts:**
1. National Abortion Comparison (side-by-side: without vs. with)
2. Top 10 Causes by Sex (stacked bars: male vs. female)
3. Abortion Impact by Race (White)
4. Abortion Impact by Race (Black/African American)

In [ ]:
import sys
import os
from pathlib import Path

import pandas as pd
import duckdb
import yaml
import altair as alt

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent / 'shared'))

from src.viz_social import save_social
from viz import PRESETS, SEX_COLORS, PALETTE

with open(PROJECT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Create outputs/social directory if needed
social_dir = PROJECT / 'outputs' / 'social'
social_dir.mkdir(parents=True, exist_ok=True)

# Connect to DuckDB
conn = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'))

# Get total abortions (national_total measure, 2024)
abort_total = conn.execute('''
  SELECT value
  FROM abortions
  WHERE measure = 'national_total' AND year = 2024
''').df()['value'].iloc[0]

# Display names for cleaner labels
DISPLAY_NAMES = {
    'Diseases of heart': 'Heart disease',
    'Malignant neoplasms': 'Cancer',
    'Chronic lower respiratory diseases': 'Respiratory disease',
    'Cerebrovascular diseases': 'Stroke',
    'Alzheimer disease': "Alzheimer's",
    'Diabetes mellitus': 'Diabetes',
    'Accidents (unintentional injuries)': 'Accidents',
    'Intentional self-harm (suicide)': 'Suicide',
    'Chronic liver disease and cirrhosis': 'Liver disease',
    'Nephritis, nephrotic syndrome and nephrosis': 'Kidney disease',
    'Influenza and pneumonia': 'Flu/Pneumonia',
    'Essential hypertension and hypertensive renal disease': 'Hypertension',
    'Assault (homicide)': 'Homicide',
    'Pregnancy, childbirth and the puerperium': 'Pregnancy/childbirth',
}

def short_name(cause: str) -> str:
    """Map verbose cause name to short display name."""
    return DISPLAY_NAMES.get(cause, cause)

print('✓ Environment loaded')
print(f'✓ Social charts will export to: {social_dir}')
print(f'✓ Total abortions 2024: {abort_total:,.0f}')

## Chart 1: National Abortion Comparison

Largest at top, no axis labels, text labels on bars, male/female colors for causes, ink_black for abortion

In [ ]:
# Get top 10 causes nationally
top_10_national = conn.execute('''
  SELECT 
    COALESCE(SUBSTR(cause, 2), cause) as cause_clean,
    deaths
  FROM mortality_national
  ORDER BY deaths DESC
  LIMIT 10
''').df()

# Apply display names
top_10_national['cause_display'] = top_10_national['cause_clean'].apply(short_name)

# Build without abortion (sorted descending - largest at top)
df_without = top_10_national[['cause_display', 'deaths']].copy()
df_without.columns = ['cause', 'deaths']
df_without['comparison'] = 'Without Abortion'
df_without['is_abortion'] = False

# Build with abortion (reranked with abortion at position 2)
df_with_list = []
rank = 0
for idx, row in top_10_national.iterrows():
    if rank == 1 and abort_total > row['deaths']:
        df_with_list.append({'cause': 'Induced Abortion', 'deaths': abort_total, 'comparison': 'With Abortion', 'is_abortion': True})
        rank += 1
    df_with_list.append({'cause': row['cause_display'], 'deaths': row['deaths'], 'comparison': 'With Abortion', 'is_abortion': False})
    rank += 1
    if rank > 11:
        break

df_with = pd.DataFrame(df_with_list)

# Combine
df_combined = pd.concat([df_without, df_with], ignore_index=True)

# Create sorted order (descending, largest at top)
cause_order = df_without.sort_values('deaths', ascending=True)['cause'].tolist()

print(f'Chart 1 data ready: {len(df_without)} causes without abortion')

In [ ]:
# Define color function based on row type
def get_bar_color(row):
    if row['is_abortion']:
        return '#003049'  # ink_black for abortion
    elif row['comparison'] == 'Without Abortion':
        return '#005F73'  # dark teal (male)
    else:
        return '#AE2012'  # oxidized red (female)

df_combined['bar_color'] = df_combined.apply(get_bar_color, axis=1)

# Build Chart 1: Side-by-side bars with no axis, text labels on right
bars = alt.Chart(df_combined).mark_bar().encode(
    y=alt.Y('cause:N', title='', sort=cause_order, 
           axis=alt.Axis(labels=True, domain=False, ticks=False, labelFontSize=11)),
    x=alt.X('deaths:Q', title='', axis=None),  # Remove x-axis entirely
    color=alt.Color('comparison:N', 
        scale=alt.Scale(
            domain=['Without Abortion', 'With Abortion'],
            range=['#005F73', '#AE2012']  # teal for without, red for with
        ),
        legend=alt.Legend(
            title=None,
            orient='top',
            labelFontSize=11,
            titleFontSize=12
        )
    ),
    xOffset='comparison:N',
).properties(
    width=1450,
    height=720,
    title={
        'text': 'If Abortion Were Counted as a Cause of Death',
        'subtitle': 'It would rank as the #2-3 leading cause in the US (2024)',
        'anchor': 'start',
        'offset': 10,
    }
)

# Add text labels to the right of bars
text = alt.Chart(df_combined).mark_text(align='left', dx=3, fontSize=10).encode(
    y=alt.Y('cause:N', sort=cause_order),
    x=alt.X('deaths:Q'),
    text=alt.Text('deaths:,.0f'),
    xOffset='comparison:N',
    color=alt.value('#374151')
)

chart1 = (bars + text).configure_axis(
    grid=False,
    domain=False,
    labelColor='#374151'
).configure_view(
    strokeWidth=0
).configure_legend(
    labelFontSize=11,
    orient='top'
)

print('Chart 1 created')
chart1

In [ ]:
# Export Chart 1
save_social(chart1, cfg, '01_abortion_comparison_national', preset='twitter_landscape')
print('✓ Chart 1 exported')

## Chart 2: Top 10 Causes by Sex

_To be updated with similar styling_

In [ ]:
# Placeholder - will be updated after Chart 1 review
print('Chart 2 placeholder - awaiting feedback on Chart 1')

## Charts 3 & 4: Abortion by Race

_To be updated with similar styling_

In [ ]:
# Placeholder - will be updated after Chart 1 review
print('Charts 3 & 4 placeholder - awaiting feedback on Chart 1')

## Cleanup

In [ ]:
# conn.close()  # Uncomment when all charts are complete
print('Ready for chart review')